#### Librerias y conexion al manejador de Base de Datos

In [20]:
import pandas as pd
import mysql.connector
from datetime import datetime

DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': '.Dur4zn05',
    'database': 'cantera_db'
}

def conectar():
    return mysql.connector.connect(**DB_CONFIG)


#Probar la conexión a la base de datos
try:    
    conexion = conectar()
    # Obtener la versión de MySQL y se imprime para confirmar la conexión
    cursor = conexion.cursor()
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()
        
    print(f"\nConexión exitosa al Manejador de Base de Datos MySQL, version: {version[0]}")
    print(f"Conectado a la Base de datos: {DB_CONFIG['database']}")
except mysql.connector.Error as err:
    print(f"\nError al conectar al Manejador de Base de Datos MySQL: {err}")




Conexión exitosa al Manejador de Base de Datos MySQL, version: 8.0.45
Conectado a la Base de datos: cantera_db


#### Funcion de Carga de CSV a Base de Datos

In [21]:
def cargar_datos_relacionados():
    conn = conectar()
    cursor = conn.cursor()
    
    if conn.is_connected():
        # 1. Cargar Jugadores (CSV Goleadores)
        df_jug = pd.read_csv('../datos/processed/Goleadores_TA25_GrupoB_clean.csv')
        filas_jugadores = 0 # Inicializamos el contador
        
        for _, fila in df_jug.iterrows():
            cursor.execute("INSERT IGNORE INTO jugadores (nombre_completo, equipo) VALUES (%s, %s)", (fila['jugador'], fila['equipo']))
            if cursor.rowcount > 0:
                filas_jugadores += cursor.rowcount # Sumamos solo si se insertó con éxito
        
        conn.commit()
        print("\n✅ Tabla de Dimensión 'Jugadores' cargada.")
        print("Filas cargadas en 'Jugadores':", filas_jugadores)
        
        
        # 2. Cargar Partidos (CSV Resultados)
        df_res = pd.read_csv('../datos/processed/Resultados_TA25_GrupoB_clean.csv')
        filas_partidos = 0 # Inicializamos el contador
        
        for _, fila in df_res.iterrows():
            eq_local, eq_vis = fila['Partido'].split(' - ')
            res = fila['Resultado'].replace('*', '').split(':')
            g_loc, g_vis = (int(res[0]), int(res[1])) if len(res) == 2 else (None, None)
            
            cursor.execute("""INSERT IGNORE INTO partidos 
                (jornada, fecha_hora, estadio, equipo_local, equipo_visitante, goles_local, goles_visitante)
                VALUES (%s, %s, %s, %s, %s, %s, %s)""",
                (fila['Jornada'], fila['Fecha/hora'], fila['Estadio'], eq_local, eq_vis, g_loc, g_vis))
            if cursor.rowcount > 0:
                filas_partidos += cursor.rowcount
        
        conn.commit()
        print("\n✅ Tabla de Dimensión 'Partidos' cargada.")
        print("Filas cargadas en 'Partidos':", filas_partidos)
        
        
        # 3. Cargar Eventos de Goles (VINCULACIÓN)
        df_eventos = pd.read_csv('../datos/processed/TablaGeneral_Goleadores_TA25_GrupoB.csv')
        filas_eventos = 0 # Inicializamos el contador
        
        for _, fila in df_eventos.iterrows():
            # Extraer fecha del nombre del archivo (ej: TA-F02-21.03.25.pdf -> 2025-03-21)
            fecha_str = fila['archivo'].split('-')[2].replace('.pdf', '')
            fecha_obj = datetime.strptime(fecha_str, '%d.%m.%y').strftime('%Y-%m-%d')
            
            # BUSCAR ID_PARTIDO: Basado en fecha y que el equipo sea local o visitante
            query_partido = """SELECT id_partido FROM partidos 
                            WHERE DATE(fecha_hora) = %s 
                            AND (equipo_local = %s OR equipo_visitante = %s) LIMIT 1"""
            cursor.execute(query_partido, (fecha_obj, fila['equipo'], fila['equipo']))
            res_p = cursor.fetchone()
            
            # BUSCAR ID_JUGADOR
            cursor.execute("SELECT id_jugador FROM jugadores WHERE nombre_completo = %s LIMIT 1", (fila['jugador'],))
            res_j = cursor.fetchone()
            
            if res_p and res_j:
                cursor.execute("""INSERT INTO eventos_goles (id_jugador, id_partido, minuto, tipo_gol, condicion)
                                VALUES (%s, %s, %s, %s, %s)""",
                            (res_j[0], res_p[0], fila['minuto'], fila['tipo_gol'], fila['condicion']))
                if cursor.rowcount > 0:
                    filas_eventos += cursor.rowcount

        conn.commit()
        print("\n✅ Tabla de Hechos 'Eventos_goles' vinculada correctamente.")
        print("Filas cargadas en 'Eventos_goles':", filas_eventos)
        
    else:
        print("\n❌ No se pudo establecer conexión a la base de datos.")
    
    cursor.close()
    conn.close()

# Cargamos los datos relacionados (jugadores, partidos y eventos de goles)
cargar_datos_relacionados()




✅ Tabla de Dimensión 'Jugadores' cargada.
Filas cargadas en 'Jugadores': 23

✅ Tabla de Dimensión 'Partidos' cargada.
Filas cargadas en 'Partidos': 42

✅ Tabla de Hechos 'Eventos_goles' vinculada correctamente.
Filas cargadas en 'Eventos_goles': 40
